In [3]:
from datasets import load_dataset

cosmopedia_v2_stream = load_dataset('HuggingFaceTB/smollm-corpus', 'cosmopedia-v2', streaming=True, split='train')
fineweb_edu_stream = load_dataset('HuggingFaceTB/smollm-corpus', 'fineweb-edu-dedup', streaming=True, split='train')
fine_math_stream = load_dataset('HuggingFaceTB/finemath', 'finemath-3plus', streaming=True, split='train')

/run/media/ocyeanic/New Volume/Transformer/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
print('----- Dataset Information----- \n')
print(cosmopedia_v2_stream)
print(fineweb_edu_stream)
print(fine_math_stream)

----- Dataset Information----- 

IterableDataset({
    features: ['prompt', 'text', 'token_length', 'audience', 'format', 'seed_data'],
    num_shards: 104
})
IterableDataset({
    features: ['text', 'id', 'metadata'],
    num_shards: 234
})
IterableDataset({
    features: ['url', 'fetch_time', 'content_mime_type', 'warc_filename', 'warc_record_offset', 'warc_record_length', 'text', 'token_count', 'char_count', 'metadata', 'score', 'int_score', 'crawl', 'snapshot_type', 'language', 'language_score'],
    num_shards: 128
})


In [5]:
import tiktoken
tokenizer = tiktoken.encoding_for_model('gpt-2')
tokenizer.decode([tokenizer.eot_token])
tokenizer

<Encoding 'gpt2'>

In [11]:
from collections.abc import Iterator, Iterable

def packer(documents:Iterable[str], tokenizer:tiktoken.core.Encoding, eos_token:int, seq_len:int):
    token_buffer = []
    for document in documents:
        doc_tokens = tokenizer.encode(document)
        doc_tokens.append(eos_token)

        token_buffer.extend(doc_tokens)

        while len(token_buffer) >= seq_len+1:
            chunk = token_buffer[:seq_len+1]
            x = chunk[:seq_len]
            y = chunk[1:seq_len]
            
            yield x, y

            del token_buffer[seq_len:]

dataset_seq_pools = {
    'fine-web-edu': packer(fineweb_edu_stream['text'], tokenizer, tokenizer.eot_token, seq_len=128),
    'cosmopedia-v2': packer(cosmopedia_v2_stream['text'], tokenizer, tokenizer.eot_token, seq_len=128),
    'fine-math': packer(fine_math_stream['text'], tokenizer, tokenizer.eot_token, seq_len=128)
}

dataset_seq_pool_iterators = {
    name: iter(pool) for name, pool in dataset_seq_pools.items()
}

def sampler(dataset_percent_dict, pool_iterators, batch_window=10, batch_size=128, total_token_budget=300000000):
    total_seq_in_pool = batch_window*batch_size
    seq_per_dataset = {name: int(percentage*total_seq_in_pool) for name, percentage in dataset_percent_dict.items()}
    token_budget_per_dataset = {name: percentage*total_token_budget for name, percentage in dataset_percent_dict.items()}
    pool = []
    for _ in range(seq_per_dataset['fine-web-edu']):
        p = pool_iterators['fine-web-edu']
        x, y = next(iter(p))
        pool.append((x, y))

    for _ in range(seq_per_dataset['cosmopedia-v2']):
        p = pool_iterators['cosmopedia-v2']
        x, y = next(iter(p))
        pool.append((x, y))

    for _ in range(seq_per_dataset['fine-math']):
        p = pool_iterators['fine-math']
        x, y = next(iter(p))
        pool.append((x, y))

    return pool


In [12]:
percent_dict = {
    'fine-web-edu': 0.40,
    'cosmopedia-v2': 0.30,
    'fine-math': 0.30,
}

pool = sampler(percent_dict, dataset_seq_pool_iterators)
len(pool)

1280